<a href="https://colab.research.google.com/github/RyuichiSaito1/inflation-reddit-usa/blob/main/notebooks/llama3_2_3b_performances_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using L4 GPU

In [ ]:
from google.colab import drive
from google.colab import auth
import os
import torch

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Authenticate Google User
auth.authenticate_user()

# 3. Install necessary libraries
# We don't need to login to Hugging Face since we use local checkpoint files.
!pip install -q accelerate transformers datasets scikit-learn matplotlib bitsandbytes

# Verify GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from torch.utils.data import Dataset
import pandas as pd

class TestDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Convert item to tensor on the fly to avoid warnings
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

def read_csv_file(file_path):
    try:
        # Read all columns as string first to prevent type inference errors
        data = pd.read_csv(file_path, names=['body', 'inflation'], header=0, dtype={'body': str, 'inflation': str})
        # Drop rows where label is missing
        data = data.dropna(subset=['inflation'])
        return data
    except FileNotFoundError:
        print(f"Error: The file at {file_path} was not found.")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

# IMF Economist Prompt (Must match training exactly)
INFLATION_PROMPT = """You are a chief economist at the IMF. I would like you to infer the public perception of inflation from Reddit posts. Please classify each Reddit post into one of the following categories: 0: The post indicates deflation, such as the lower price of goods or services (e.g., "the prices are not bad"), affordable services (e.g., "this champagne is cheap and delicious"), sales information (e.g., "you can get it for only 10 dollars."), or a declining and buyer's market. 2: The post indicates or includes inflation, such as the higher price of goods or services (e.g., "it's not cheap"), the unreasonable cost of goods or services (e.g., "the food is overpriced and cold"), consumers struggling to afford necessities (e.g., "items are too expensive to buy"), shortage of goods of services, or mention about an asset bubble. 1: The post indicates neither deflation (0) nor inflation (2). This category also includes just questions to a community, social statements not personal experience, factual observations, references to originally expensive or cheap goods or services (e.g., "a gorgeous and costly dinner" or "an affordable Civic"), website promotion, authors' wishes, or illogical text. Please choose a stronger stance when the text includes both 0 and 2 stances. If these stances are of the same degree, answer 1.

Reddit Post: {post}

Classification:"""

def format_with_prompt(post):
    return INFLATION_PROMPT.format(post=str(post))

In [ ]:

# Path to test data
file_path = '/content/drive/MyDrive/world-inflation/data/reddit/production/test-data-200.csv'

# Read data
test_data = read_csv_file(file_path)

if test_data is not None:
    # Apply prompt formatting
    test_data['formatted_body'] = test_data['body'].apply(format_with_prompt)
    print(f"Data loaded successfully. Number of samples: {len(test_data)}")

    # Verify the first sample
    print("\nSample input:")
    print(test_data['formatted_body'].iloc[0][:200] + "...")
else:
    raise ValueError("Failed to load test data.")

# SFT: 1,039

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Checkpoint path (Updated to 147)
checkpoint_path = '/content/drive/MyDrive/world-inflation/data/model/llama-3.2-3b-fine-tuning-rerun/checkpoint-147/'

print(f"Loading tokenizer and model from: {checkpoint_path}")

# Load Tokenizer from local files
base_model_name = "meta-llama/Llama-3.2-3B"
print(f"Loading tokenizer from base model: {base_model_name}")

try:
    # Use base_model_name instead of checkpoint_path
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, token=True)
except OSError as e:
    print(f"Error loading tokenizer: {e}")
    raise

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load Model from local files
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        num_labels=3,
        local_files_only=True
    )
except OSError as e:
    print(f"Error loading model: {e}")
    raise

model.eval()
print("Model loaded successfully.")

In [ ]:
from torch.utils.data import DataLoader

# 1. Tokenize
print("Tokenizing test data...")
test_encodings = tokenizer(
    test_data['formatted_body'].tolist(),
    truncation=True,
    padding=True,
    max_length=512
)

# 2. Prepare Labels
try:
    test_labels = test_data['inflation'].astype(int).tolist()
except ValueError:
    # Clean non-numeric labels if necessary
    print("Warning: Found non-numeric labels. attempting to coerce...")
    test_data['inflation'] = pd.to_numeric(test_data['inflation'], errors='coerce')
    test_data = test_data.dropna(subset=['inflation'])
    test_labels = test_data['inflation'].astype(int).tolist()
    # Re-tokenize might be needed if rows were dropped, but assuming clean data for now

# 3. Create DataLoader
test_dataset = TestDataset(test_encodings, test_labels)
test_loader = DataLoader(test_dataset, batch_size=8)

# 4. Inference Loop
print("Starting inference...")
true_labels = []
predicted_labels = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        # Move inputs to device
        inputs = {key: val.to(model.device) for key, val in batch.items() if key != 'labels'}
        labels = batch['labels'].to(model.device)

        # Forward pass
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)

        # Store results
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

        # Logging
        if (batch_idx + 1) % 5 == 0:
            print(f"Processed {(batch_idx + 1) * 8} / {len(test_dataset)} samples...")

print("Inference completed.")

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report, confusion_matrix
import numpy as np

# Calculate Metrics
accuracy = accuracy_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels, average=None)
precision = precision_score(true_labels, predicted_labels, average=None)
f1 = f1_score(true_labels, predicted_labels, average=None)

# Display Classification Report
print("\n=== Classification Report ===")
print(classification_report(true_labels, predicted_labels, digits=4))

# Display Confusion Matrix
print("\n=== Confusion Matrix ===")
print(confusion_matrix(true_labels, predicted_labels))

# Display Detailed Metrics Table
print("\n=== Detailed Metrics ===")
print("+--------------+-----------+----------+----------+----------+")
print("|   Metric     | Accuracy  |  Recall  | Precision|  F1 Score |")
print("+--------------+-----------+----------+----------+----------+")
for i in range(len(recall)):
    print(f"| Class {i}      |    ---    |   {recall[i]:.4f} |   {precision[i]:.4f} |   {f1[i]:.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Macro Average
print(f"| Macro Average|    ---    |   {np.mean(recall):.4f} |   {np.mean(precision):.4f} |   {np.mean(f1):.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Micro Average (Same as Accuracy for single-label)
print(f"| Micro Average|   {accuracy:.4f}  |   {accuracy:.4f} |   {accuracy:.4f} |   {accuracy:.4f} |")
print("+--------------+-----------+----------+----------+----------+")

print(f"\nTotal test samples: {len(true_labels)}")

# SFT 520

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Checkpoint path
# ☆
checkpoint_path = '/content/drive/MyDrive/world-inflation/data/model/llama-3.2-3b-fine-tuning-520-rerun'

print(f"Loading tokenizer and model from: {checkpoint_path}")

# Load Tokenizer from local files
# try:
#     tokenizer = AutoTokenizer.from_pretrained(checkpoint_path, local_files_only=True)
# except OSError as e:
#     print(f"Error loading tokenizer: {e}")
#     print("Please verify that tokenizer.json and config files exist in the checkpoint folder.")
#     raise

base_model_name = "meta-llama/Llama-3.2-3B"
print(f"Loading tokenizer from: {base_model_name}")

tokenizer = AutoTokenizer.from_pretrained(base_model_name, token=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load Model from local files
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        num_labels=3,
        local_files_only=True
    )
except OSError as e:
    print(f"Error loading model: {e}")
    raise

model.eval()
print("Model loaded successfully.")

from torch.utils.data import DataLoader

# 1. Tokenize
print("Tokenizing test data...")
test_encodings = tokenizer(
    test_data['formatted_body'].tolist(),
    truncation=True,
    padding=True,
    max_length=512
)

# 2. Prepare Labels
try:
    test_labels = test_data['inflation'].astype(int).tolist()
except ValueError:
    # Clean non-numeric labels if necessary
    print("Warning: Found non-numeric labels. attempting to coerce...")
    test_data['inflation'] = pd.to_numeric(test_data['inflation'], errors='coerce')
    test_data = test_data.dropna(subset=['inflation'])
    test_labels = test_data['inflation'].astype(int).tolist()
    # Re-tokenize might be needed if rows were dropped, but assuming clean data for now

# 3. Create DataLoader
test_dataset = TestDataset(test_encodings, test_labels)
test_loader = DataLoader(test_dataset, batch_size=8)

# 4. Inference Loop
print("Starting inference...")
true_labels = []
predicted_labels = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        # Move inputs to device
        inputs = {key: val.to(model.device) for key, val in batch.items() if key != 'labels'}
        labels = batch['labels'].to(model.device)

        # Forward pass
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)

        # Store results
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

        # Logging
        if (batch_idx + 1) % 5 == 0:
            print(f"Processed {(batch_idx + 1) * 8} / {len(test_dataset)} samples...")

print("Inference completed.")

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report, confusion_matrix
import numpy as np

# Calculate Metrics
accuracy = accuracy_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels, average=None)
precision = precision_score(true_labels, predicted_labels, average=None)
f1 = f1_score(true_labels, predicted_labels, average=None)

# Display Classification Report
print("\n=== Classification Report ===")
print(classification_report(true_labels, predicted_labels, digits=4))

# Display Confusion Matrix
print("\n=== Confusion Matrix ===")
print(confusion_matrix(true_labels, predicted_labels))

# Display Detailed Metrics Table
print("\n=== Detailed Metrics ===")
print("+--------------+-----------+----------+----------+----------+")
print("|   Metric     | Accuracy  |  Recall  | Precision|  F1 Score |")
print("+--------------+-----------+----------+----------+----------+")
for i in range(len(recall)):
    print(f"| Class {i}      |    ---    |   {recall[i]:.4f} |   {precision[i]:.4f} |   {f1[i]:.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Macro Average
print(f"| Macro Average|    ---    |   {np.mean(recall):.4f} |   {np.mean(precision):.4f} |   {np.mean(f1):.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Micro Average (Same as Accuracy for single-label)
print(f"| Micro Average|   {accuracy:.4f}  |   {accuracy:.4f} |   {accuracy:.4f} |   {accuracy:.4f} |")
print("+--------------+-----------+----------+----------+----------+")

print(f"\nTotal test samples: {len(true_labels)}")

# SFT 260

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Checkpoint path
# ☆
checkpoint_path = '/content/drive/MyDrive/world-inflation/data/model/llama-3.2-3b-fine-tuning-260-rerun'

print(f"Loading tokenizer and model from: {checkpoint_path}")

base_model_name = "meta-llama/Llama-3.2-3B"
print(f"Loading tokenizer from: {base_model_name}")

tokenizer = AutoTokenizer.from_pretrained(base_model_name, token=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load Model from local files
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        num_labels=3,
        local_files_only=True
    )
except OSError as e:
    print(f"Error loading model: {e}")
    raise

model.eval()
print("Model loaded successfully.")

from torch.utils.data import DataLoader

# 1. Tokenize
print("Tokenizing test data...")
test_encodings = tokenizer(
    test_data['formatted_body'].tolist(),
    truncation=True,
    padding=True,
    max_length=512
)

# 2. Prepare Labels
try:
    test_labels = test_data['inflation'].astype(int).tolist()
except ValueError:
    # Clean non-numeric labels if necessary
    print("Warning: Found non-numeric labels. attempting to coerce...")
    test_data['inflation'] = pd.to_numeric(test_data['inflation'], errors='coerce')
    test_data = test_data.dropna(subset=['inflation'])
    test_labels = test_data['inflation'].astype(int).tolist()
    # Re-tokenize might be needed if rows were dropped, but assuming clean data for now

# 3. Create DataLoader
test_dataset = TestDataset(test_encodings, test_labels)
test_loader = DataLoader(test_dataset, batch_size=8)

# 4. Inference Loop
print("Starting inference...")
true_labels = []
predicted_labels = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        # Move inputs to device
        inputs = {key: val.to(model.device) for key, val in batch.items() if key != 'labels'}
        labels = batch['labels'].to(model.device)

        # Forward pass
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)

        # Store results
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

        # Logging
        if (batch_idx + 1) % 5 == 0:
            print(f"Processed {(batch_idx + 1) * 8} / {len(test_dataset)} samples...")

print("Inference completed.")

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report, confusion_matrix
import numpy as np

# Calculate Metrics
accuracy = accuracy_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels, average=None)
precision = precision_score(true_labels, predicted_labels, average=None)
f1 = f1_score(true_labels, predicted_labels, average=None)

# Display Classification Report
print("\n=== Classification Report ===")
print(classification_report(true_labels, predicted_labels, digits=4))

# Display Confusion Matrix
print("\n=== Confusion Matrix ===")
print(confusion_matrix(true_labels, predicted_labels))

# Display Detailed Metrics Table
print("\n=== Detailed Metrics ===")
print("+--------------+-----------+----------+----------+----------+")
print("|   Metric     | Accuracy  |  Recall  | Precision|  F1 Score |")
print("+--------------+-----------+----------+----------+----------+")
for i in range(len(recall)):
    print(f"| Class {i}      |    ---    |   {recall[i]:.4f} |   {precision[i]:.4f} |   {f1[i]:.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Macro Average
print(f"| Macro Average|    ---    |   {np.mean(recall):.4f} |   {np.mean(precision):.4f} |   {np.mean(f1):.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Micro Average (Same as Accuracy for single-label)
print(f"| Micro Average|   {accuracy:.4f}  |   {accuracy:.4f} |   {accuracy:.4f} |   {accuracy:.4f} |")
print("+--------------+-----------+----------+----------+----------+")

print(f"\nTotal test samples: {len(true_labels)}")

# 130

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Checkpoint path
# ☆
checkpoint_path = '/content/drive/MyDrive/world-inflation/data/model/llama-3.2-3b-fine-tuning-130-rerun'

print(f"Loading tokenizer and model from: {checkpoint_path}")

base_model_name = "meta-llama/Llama-3.2-3B"
print(f"Loading tokenizer from: {base_model_name}")

tokenizer = AutoTokenizer.from_pretrained(base_model_name, token=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load Model from local files
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        num_labels=3,
        local_files_only=True
    )
except OSError as e:
    print(f"Error loading model: {e}")
    raise

model.eval()
print("Model loaded successfully.")

from torch.utils.data import DataLoader

# 1. Tokenize
print("Tokenizing test data...")
test_encodings = tokenizer(
    test_data['formatted_body'].tolist(),
    truncation=True,
    padding=True,
    max_length=512
)

# 2. Prepare Labels
try:
    test_labels = test_data['inflation'].astype(int).tolist()
except ValueError:
    # Clean non-numeric labels if necessary
    print("Warning: Found non-numeric labels. attempting to coerce...")
    test_data['inflation'] = pd.to_numeric(test_data['inflation'], errors='coerce')
    test_data = test_data.dropna(subset=['inflation'])
    test_labels = test_data['inflation'].astype(int).tolist()
    # Re-tokenize might be needed if rows were dropped, but assuming clean data for now

# 3. Create DataLoader
test_dataset = TestDataset(test_encodings, test_labels)
test_loader = DataLoader(test_dataset, batch_size=8)

# 4. Inference Loop
print("Starting inference...")
true_labels = []
predicted_labels = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        # Move inputs to device
        inputs = {key: val.to(model.device) for key, val in batch.items() if key != 'labels'}
        labels = batch['labels'].to(model.device)

        # Forward pass
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)

        # Store results
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

        # Logging
        if (batch_idx + 1) % 5 == 0:
            print(f"Processed {(batch_idx + 1) * 8} / {len(test_dataset)} samples...")

print("Inference completed.")

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report, confusion_matrix
import numpy as np

# Calculate Metrics
accuracy = accuracy_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels, average=None)
precision = precision_score(true_labels, predicted_labels, average=None)
f1 = f1_score(true_labels, predicted_labels, average=None)

# Display Classification Report
print("\n=== Classification Report ===")
print(classification_report(true_labels, predicted_labels, digits=4))

# Display Confusion Matrix
print("\n=== Confusion Matrix ===")
print(confusion_matrix(true_labels, predicted_labels))

# Display Detailed Metrics Table
print("\n=== Detailed Metrics ===")
print("+--------------+-----------+----------+----------+----------+")
print("|   Metric     | Accuracy  |  Recall  | Precision|  F1 Score |")
print("+--------------+-----------+----------+----------+----------+")
for i in range(len(recall)):
    print(f"| Class {i}      |    ---    |   {recall[i]:.4f} |   {precision[i]:.4f} |   {f1[i]:.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Macro Average
print(f"| Macro Average|    ---    |   {np.mean(recall):.4f} |   {np.mean(precision):.4f} |   {np.mean(f1):.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Micro Average (Same as Accuracy for single-label)
print(f"| Micro Average|   {accuracy:.4f}  |   {accuracy:.4f} |   {accuracy:.4f} |   {accuracy:.4f} |")
print("+--------------+-----------+----------+----------+----------+")

print(f"\nTotal test samples: {len(true_labels)}")

# 65

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Checkpoint path
# ☆
checkpoint_path = '/content/drive/MyDrive/world-inflation/data/model/llama-3.2-3b-fine-tuning-65-rerun/final_best'

print(f"Loading tokenizer and model from: {checkpoint_path}")

base_model_name = "meta-llama/Llama-3.2-3B"
print(f"Loading tokenizer from: {base_model_name}")

tokenizer = AutoTokenizer.from_pretrained(base_model_name, token=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load Model from local files
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        num_labels=3,
        local_files_only=True
    )
except OSError as e:
    print(f"Error loading model: {e}")
    raise

model.eval()
print("Model loaded successfully.")

from torch.utils.data import DataLoader

# 1. Tokenize
print("Tokenizing test data...")
test_encodings = tokenizer(
    test_data['formatted_body'].tolist(),
    truncation=True,
    padding=True,
    max_length=512
)

# 2. Prepare Labels
try:
    test_labels = test_data['inflation'].astype(int).tolist()
except ValueError:
    # Clean non-numeric labels if necessary
    print("Warning: Found non-numeric labels. attempting to coerce...")
    test_data['inflation'] = pd.to_numeric(test_data['inflation'], errors='coerce')
    test_data = test_data.dropna(subset=['inflation'])
    test_labels = test_data['inflation'].astype(int).tolist()
    # Re-tokenize might be needed if rows were dropped, but assuming clean data for now

# 3. Create DataLoader
test_dataset = TestDataset(test_encodings, test_labels)
test_loader = DataLoader(test_dataset, batch_size=8)

# 4. Inference Loop
print("Starting inference...")
true_labels = []
predicted_labels = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        # Move inputs to device
        inputs = {key: val.to(model.device) for key, val in batch.items() if key != 'labels'}
        labels = batch['labels'].to(model.device)

        # Forward pass
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)

        # Store results
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

        # Logging
        if (batch_idx + 1) % 5 == 0:
            print(f"Processed {(batch_idx + 1) * 8} / {len(test_dataset)} samples...")

print("Inference completed.")

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report, confusion_matrix
import numpy as np

# Calculate Metrics
accuracy = accuracy_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels, average=None)
precision = precision_score(true_labels, predicted_labels, average=None)
f1 = f1_score(true_labels, predicted_labels, average=None)

# Display Classification Report
print("\n=== Classification Report ===")
print(classification_report(true_labels, predicted_labels, digits=4))

# Display Confusion Matrix
print("\n=== Confusion Matrix ===")
print(confusion_matrix(true_labels, predicted_labels))

# Display Detailed Metrics Table
print("\n=== Detailed Metrics ===")
print("+--------------+-----------+----------+----------+----------+")
print("|   Metric     | Accuracy  |  Recall  | Precision|  F1 Score |")
print("+--------------+-----------+----------+----------+----------+")
for i in range(len(recall)):
    print(f"| Class {i}      |    ---    |   {recall[i]:.4f} |   {precision[i]:.4f} |   {f1[i]:.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Macro Average
print(f"| Macro Average|    ---    |   {np.mean(recall):.4f} |   {np.mean(precision):.4f} |   {np.mean(f1):.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Micro Average (Same as Accuracy for single-label)
print(f"| Micro Average|   {accuracy:.4f}  |   {accuracy:.4f} |   {accuracy:.4f} |   {accuracy:.4f} |")
print("+--------------+-----------+----------+----------+----------+")

print(f"\nTotal test samples: {len(true_labels)}")